# CS4.406 A2, Q2 (before/after), Q3 (baseline and ablation), Q5 (extended evaluation)

Input: `reranked_ebnerd_validation_scores.parquet` and `reranked_mind_validation_scores.parquet`, produced by the retrieval and re-ranking half of the pipeline. Each row is one impression with the candidate list, ground truth labels, the stage 1 score (retrieval only), and the stage 2 score (re-ranked).

Baseline for Q3 is stage 1 (retrieval score, no click log signal). Improved is stage 2 (the trained re-ranker

In [78]:
import numpy as np
import polars as pl
from pathlib import Path
from sklearn.metrics import roc_auc_score

def find_one(pattern):
    hits = sorted(Path("/kaggle/input").rglob(pattern))
    if not hits:
        raise FileNotFoundError(f"could not find {pattern} under /kaggle/input")
    return hits[0]

def find_by_folder(folder_keyword, filename_pattern):
    hits = sorted(p for p in Path("/kaggle/input").rglob(filename_pattern)
                  if folder_keyword in p.parent.name)
    if not hits:
        raise FileNotFoundError(f"could not find {filename_pattern} in a folder containing '{folder_keyword}'")
    return hits[0]

def find_by_dataset(dataset_keyword, filename_pattern):
    hits = sorted(p for p in Path("/kaggle/input").rglob(filename_pattern)
                  if dataset_keyword in str(p))
    if not hits:
        raise FileNotFoundError(f"could not find {filename_pattern} under a path containing '{dataset_keyword}'")
    return hits[0]

DATASETS = {
    "EB-NeRD": find_one("reranked_ebnerd_validation_scores.parquet"),
    "MIND": find_one("reranked_mind_validation_scores.parquet"),
}
dfs = {name: pl.read_parquet(path) for name, path in DATASETS.items()}
for name, df in dfs.items():
    print(name, df.shape, df.columns)


EB-NeRD (5000, 6) ['impression_id', 'user_id', 'candidate_ids', 'labels', 'stage1_scores', 'reranked_scores']
MIND (5000, 6) ['impression_id', 'user_id', 'candidate_ids', 'labels', 'stage1_scores', 'reranked_scores']


## Q2. Before and after re-ranking

In [79]:
def auc_per_impression(labels, scores):
    labels = np.asarray(labels)
    if labels.min() == labels.max():
        return np.nan
    return roc_auc_score(labels, scores)

def mrr_per_impression(labels, scores):
    order = np.argsort(-np.asarray(scores))
    ranked = np.asarray(labels)[order]
    hits = np.flatnonzero(ranked == 1)
    return np.nan if len(hits) == 0 else 1.0 / (hits[0] + 1)

def _dcg_at_k(labels_sorted, k):
    labels_sorted = np.asarray(labels_sorted[:k], dtype=np.float64)
    discounts = 1.0 / np.log2(np.arange(2, len(labels_sorted) + 2))
    return float(np.sum(labels_sorted * discounts))

def ndcg_at_k(labels, scores, k):
    labels = np.asarray(labels)
    order = np.argsort(-np.asarray(scores))
    dcg = _dcg_at_k(labels[order], k)
    idcg = _dcg_at_k(np.sort(labels)[::-1], k)
    return np.nan if idcg == 0 else dcg / idcg

def per_impression_metrics(df, score_col):
    aucs, mrrs, ndcg5, ndcg10 = [], [], [], []
    for labels, scores in zip(df["labels"].to_list(), df[score_col].to_list()):
        aucs.append(auc_per_impression(labels, scores))
        mrrs.append(mrr_per_impression(labels, scores))
        ndcg5.append(ndcg_at_k(labels, scores, 5))
        ndcg10.append(ndcg_at_k(labels, scores, 10))
    return dict(auc=np.array(aucs), mrr=np.array(mrrs),
                ndcg5=np.array(ndcg5), ndcg10=np.array(ndcg10))

metrics = {}
for name, df in dfs.items():
    metrics[name] = {
        "stage1": per_impression_metrics(df, "stage1_scores"),
        "stage2": per_impression_metrics(df, "reranked_scores"),
    }
print("per impression metrics computed for both stages, both datasets")


per impression metrics computed for both stages, both datasets


In [80]:
before_after_rows = []
for name in DATASETS:
    for metric_name in ("auc", "mrr", "ndcg5", "ndcg10"):
        before = np.nanmean(metrics[name]["stage1"][metric_name])
        after = np.nanmean(metrics[name]["stage2"][metric_name])
        before_after_rows.append({
            "dataset": name, "metric": metric_name,
            "before_rerank": round(before, 4), "after_rerank": round(after, 4),
            "delta": round(after - before, 4),
        })

before_after = pl.DataFrame(before_after_rows)
print(before_after)
before_after.write_csv("q2_before_after_rerank.csv")


shape: (8, 5)
┌─────────┬────────┬───────────────┬──────────────┬────────┐
│ dataset ┆ metric ┆ before_rerank ┆ after_rerank ┆ delta  │
│ ---     ┆ ---    ┆ ---           ┆ ---          ┆ ---    │
│ str     ┆ str    ┆ f64           ┆ f64          ┆ f64    │
╞═════════╪════════╪═══════════════╪══════════════╪════════╡
│ EB-NeRD ┆ auc    ┆ 0.4846        ┆ 0.6957       ┆ 0.2111 │
│ EB-NeRD ┆ mrr    ┆ 0.3069        ┆ 0.4578       ┆ 0.1509 │
│ EB-NeRD ┆ ndcg5  ┆ 0.3331        ┆ 0.5172       ┆ 0.1841 │
│ EB-NeRD ┆ ndcg10 ┆ 0.4227        ┆ 0.5683       ┆ 0.1456 │
│ MIND    ┆ auc    ┆ 0.561         ┆ 0.6067       ┆ 0.0457 │
│ MIND    ┆ mrr    ┆ 0.3038        ┆ 0.3264       ┆ 0.0226 │
│ MIND    ┆ ndcg5  ┆ 0.2782        ┆ 0.3025       ┆ 0.0243 │
│ MIND    ┆ ndcg10 ┆ 0.342         ┆ 0.3669       ┆ 0.0249 │
└─────────┴────────┴───────────────┴──────────────┴────────┘


Observations, Q2:
- Re-ranking improves all four metrics on both datasets. No metric goes down after stage 2.
- The improvement is larger on EB-NeRD than on MIND in relative terms.


## Q3. Baseline reproduced, then beaten, with paired bootstrap CI

In [81]:
def paired_bootstrap_ci_diff(arr_improved, arr_baseline, n_boot=2000, seed=0):
    a, b = np.asarray(arr_improved), np.asarray(arr_baseline)
    mask = ~(np.isnan(a) | np.isnan(b))
    a, b = a[mask], b[mask]
    n = len(a)
    rng = np.random.default_rng(seed)
    diffs = np.empty(n_boot)
    for i in range(n_boot):
        idx = rng.integers(0, n, size=n)
        diffs[i] = a[idx].mean() - b[idx].mean()
    lo, hi = np.percentile(diffs, [2.5, 97.5])
    return float(diffs.mean()), float(lo), float(hi), n

ablation_rows = []
for name in DATASETS:
    for metric_name in ("auc", "mrr", "ndcg5", "ndcg10"):
        improved = metrics[name]["stage2"][metric_name]
        baseline = metrics[name]["stage1"][metric_name]
        mean_diff, lo, hi, n_used = paired_bootstrap_ci_diff(improved, baseline)
        significant = lo > 0
        ablation_rows.append({
            "dataset": name, "metric": metric_name, "n_impressions": n_used,
            "mean_delta": round(mean_diff, 4), "ci_lo": round(lo, 4), "ci_hi": round(hi, 4),
            "significant": significant,
        })

ablation_table = pl.DataFrame(ablation_rows)
print(ablation_table)
ablation_table.write_csv("q3_ablation_paired_ci.csv")


shape: (8, 7)
┌─────────┬────────┬───────────────┬────────────┬────────┬────────┬─────────────┐
│ dataset ┆ metric ┆ n_impressions ┆ mean_delta ┆ ci_lo  ┆ ci_hi  ┆ significant │
│ ---     ┆ ---    ┆ ---           ┆ ---        ┆ ---    ┆ ---    ┆ ---         │
│ str     ┆ str    ┆ i64           ┆ f64        ┆ f64    ┆ f64    ┆ bool        │
╞═════════╪════════╪═══════════════╪════════════╪════════╪════════╪═════════════╡
│ EB-NeRD ┆ auc    ┆ 5000          ┆ 0.2111     ┆ 0.1994 ┆ 0.2232 ┆ true        │
│ EB-NeRD ┆ mrr    ┆ 5000          ┆ 0.1508     ┆ 0.1393 ┆ 0.162  ┆ true        │
│ EB-NeRD ┆ ndcg5  ┆ 5000          ┆ 0.1839     ┆ 0.1714 ┆ 0.1963 ┆ true        │
│ EB-NeRD ┆ ndcg10 ┆ 5000          ┆ 0.1456     ┆ 0.1359 ┆ 0.1551 ┆ true        │
│ MIND    ┆ auc    ┆ 5000          ┆ 0.0457     ┆ 0.0387 ┆ 0.0524 ┆ true        │
│ MIND    ┆ mrr    ┆ 5000          ┆ 0.0227     ┆ 0.0169 ┆ 0.0288 ┆ true        │
│ MIND    ┆ ndcg5  ┆ 5000          ┆ 0.0243     ┆ 0.0186 ┆ 0.0302 ┆ true        │
│ 

Observations, Q3:
- EB-NeRD: AUC delta +0.211, CI [+0.199, +0.223]. MRR +0.151, nDCG@5 +0.184, nDCG@10 +0.146. All CIs exclude zero.
- MIND: AUC delta +0.046, CI [+0.039, +0.052]. MRR +0.023, nDCG@5 +0.024, nDCG@10 +0.025. All CIs exclude zero.
- Every metric on both datasets is significant at the 95 percent level. The gain is larger on EB-NeRD, consistent with its shorter candidate lists making rank-based signal sharper relative to the baseline.


## Q5. Extended evaluation

In [82]:
def bootstrap_ci(values, n_boot=1000, seed=0):
    values = values[~np.isnan(values)]
    rng = np.random.default_rng(seed)
    n = len(values)
    boot_means = np.array([rng.choice(values, size=n, replace=True).mean() for _ in range(n_boot)])
    lo, hi = np.percentile(boot_means, [2.5, 97.5])
    return float(values.mean()), float(lo), float(hi)

rows = []
for name in DATASETS:
    for stage in ("stage1", "stage2"):
        for metric_name, arr in metrics[name][stage].items():
            mean, lo, hi = bootstrap_ci(arr)
            rows.append({"dataset": name, "stage": stage, "metric": metric_name,
                          "mean": round(mean, 4), "ci_lo": round(lo, 4), "ci_hi": round(hi, 4)})

summary_table = pl.DataFrame(rows)
print(summary_table)
summary_table.write_csv("q5_metrics_summary.csv")


shape: (16, 6)
┌─────────┬────────┬────────┬────────┬────────┬────────┐
│ dataset ┆ stage  ┆ metric ┆ mean   ┆ ci_lo  ┆ ci_hi  │
│ ---     ┆ ---    ┆ ---    ┆ ---    ┆ ---    ┆ ---    │
│ str     ┆ str    ┆ str    ┆ f64    ┆ f64    ┆ f64    │
╞═════════╪════════╪════════╪════════╪════════╪════════╡
│ EB-NeRD ┆ stage1 ┆ auc    ┆ 0.4846 ┆ 0.4758 ┆ 0.4939 │
│ EB-NeRD ┆ stage1 ┆ mrr    ┆ 0.3069 ┆ 0.2997 ┆ 0.3146 │
│ EB-NeRD ┆ stage1 ┆ ndcg5  ┆ 0.3331 ┆ 0.3241 ┆ 0.3424 │
│ EB-NeRD ┆ stage1 ┆ ndcg10 ┆ 0.4227 ┆ 0.4156 ┆ 0.4302 │
│ EB-NeRD ┆ stage2 ┆ auc    ┆ 0.6957 ┆ 0.6881 ┆ 0.7036 │
│ …       ┆ …      ┆ …      ┆ …      ┆ …      ┆ …      │
│ MIND    ┆ stage1 ┆ ndcg10 ┆ 0.342  ┆ 0.3335 ┆ 0.3516 │
│ MIND    ┆ stage2 ┆ auc    ┆ 0.6067 ┆ 0.5991 ┆ 0.6152 │
│ MIND    ┆ stage2 ┆ mrr    ┆ 0.3264 ┆ 0.3176 ┆ 0.3366 │
│ MIND    ┆ stage2 ┆ ndcg5  ┆ 0.3025 ┆ 0.2935 ┆ 0.3132 │
│ MIND    ┆ stage2 ┆ ndcg10 ┆ 0.3669 ┆ 0.3583 ┆ 0.3767 │
└─────────┴────────┴────────┴────────┴────────┴────────┘


Observations, metrics summary:
- Stage 2 confidence intervals sit above stage 1 on every metric, both datasets. No overlap between stage 1 and stage 2 intervals.


### Article and user metadata, EB-NeRD

In [83]:
behaviors_paths = [find_by_folder("train", "behaviors.parquet"),
                    find_by_folder("validation", "behaviors.parquet")]
history_paths = [find_by_folder("train", "history.parquet"),
                  find_by_folder("validation", "history.parquet")]

articles = pl.read_parquet(find_one("articles.parquet"))
cat_col = "category_str" if "category_str" in articles.columns else "category"

clicks = (
    pl.concat([pl.scan_parquet(p) for p in behaviors_paths])
    .select("article_ids_clicked")
    .explode("article_ids_clicked")
    .drop_nulls()
    .group_by("article_ids_clicked")
    .agg(pl.len().alias("click_count"))
    .collect()
)

article_meta = (
    articles.select(["article_id", cat_col])
    .rename({cat_col: "category"})
    .with_columns(pl.col("article_id").cast(pl.Utf8))
    .join(clicks.with_columns(pl.col("article_ids_clicked").cast(pl.Utf8)),
          left_on="article_id", right_on="article_ids_clicked", how="left")
    .with_columns(pl.col("click_count").fill_null(0))
    .with_columns((pl.col("click_count") / pl.col("click_count").sum()).alias("popularity"))
    .select(["article_id", "category", "popularity"])
)

hist_all = pl.concat([pl.read_parquet(p) for p in history_paths])
user_hist_len = (
    hist_all.select(["user_id", "article_id_fixed"])
    .with_columns(pl.col("article_id_fixed").list.len().alias("hist_len"))
    .unique(subset="user_id", keep="last")
    .with_columns(pl.col("user_id").cast(pl.Utf8))
    .select(["user_id", "hist_len"])
)

all_candidates_ebnerd = dfs["EB-NeRD"]["candidate_ids"].explode().unique().to_list()
pop_lookup = dict(zip(article_meta["article_id"], article_meta["popularity"]))
n_covered = sum(1 for a in all_candidates_ebnerd if pop_lookup.get(a, 0.0) > 0)
print(f"{n_covered} / {len(all_candidates_ebnerd)} candidate articles have nonzero popularity "
      f"({100*n_covered/len(all_candidates_ebnerd):.1f}%)")

article_meta.write_parquet("article_meta.parquet")
user_hist_len.write_parquet("user_hist_len.parquet")


1916 / 1934 candidate articles have nonzero popularity (99.1%)


Observations, EB-NeRD metadata:
- 99.1 percent of candidate articles that appear in the evaluation set have nonzero popularity. Coverage is effectively complete for the articles that matter, even though most of the full catalogue is dormant (published outside the click log window).


### Article and user metadata, MIND

In [84]:
news_dev = pl.read_csv(find_by_folder("dev", "news.tsv"), separator="\t", has_header=False,
                        columns=[0, 1], new_columns=["article_id", "category"])
news_train = pl.read_csv(find_by_dataset("mind-large-train", "news.tsv"), separator="\t",
                          has_header=False, columns=[0, 1], new_columns=["article_id", "category"])
news_small_dev = pl.read_csv(find_by_dataset("mind-small-dev", "news.tsv"), separator="\t",
                              has_header=False, columns=[0, 1], new_columns=["article_id", "category"])
news_test = pl.read_csv(find_by_dataset("mind-large-test", "news.tsv"), separator="\t",
                         has_header=False, columns=[0, 1], new_columns=["article_id", "category"])
news_small_train = pl.read_csv(find_by_dataset("arashnic", "news.tsv"), separator="\t",
                                has_header=False, columns=[0, 1], new_columns=["article_id", "category"])

news_full = pl.concat([news_dev, news_train, news_small_dev, news_test, news_small_train]).unique(
    subset="article_id", keep="first")

all_candidates_mind = dfs["MIND"]["candidate_ids"].explode().unique()
all_candidates_list = all_candidates_mind.to_list()
news_ids = set(news_full["article_id"].to_list())
n_in_news = sum(1 for a in all_candidates_list if a in news_ids)
print(f"{news_full.height} total distinct articles across all five MIND sources")
print(f"{n_in_news} / {len(all_candidates_list)} candidate articles exist in the combined catalogue "
      f"({100*n_in_news/len(all_candidates_list):.1f}%)")

def extract_clicks(beh_df):
    return (
        beh_df.select("impressions")
        .with_columns(pl.col("impressions").str.split(" "))
        .explode("impressions")
        .filter(pl.col("impressions").str.ends_with("-1"))
        .with_columns(pl.col("impressions").str.replace("-1", "").alias("article_id"))
        .group_by("article_id")
        .agg(pl.len().alias("click_count"))
    )

behaviors_dev = pl.read_csv(find_by_folder("dev", "behaviors.tsv"), separator="\t", has_header=False,
                             new_columns=["imp_id", "user_id", "time", "history", "impressions"])
behaviors_train = pl.read_csv(find_by_dataset("mind-large-train", "behaviors.tsv"), separator="\t",
                               has_header=False,
                               new_columns=["imp_id", "user_id", "time", "history", "impressions"])
behaviors_small_dev = pl.read_csv(find_by_dataset("mind-small-dev", "behaviors.tsv"), separator="\t",
                                   has_header=False,
                                   new_columns=["imp_id", "user_id", "time", "history", "impressions"])

clicked_dev = extract_clicks(behaviors_dev)
clicked_train = extract_clicks(behaviors_train)
clicked_small_dev = extract_clicks(behaviors_small_dev)

clicked_combined = (
    pl.concat([clicked_dev, clicked_train, clicked_small_dev])
    .group_by("article_id")
    .agg(pl.col("click_count").sum())
)

article_meta_mind = (
    news_full.join(clicked_combined, on="article_id", how="left")
    .with_columns(pl.col("click_count").fill_null(0))
    .with_columns((pl.col("click_count") / pl.col("click_count").sum()).alias("popularity"))
    .select(["article_id", "category", "popularity"])
)

pop_lookup_mind = dict(zip(article_meta_mind["article_id"], article_meta_mind["popularity"]))
n_covered_mind = sum(1 for a in all_candidates_list if pop_lookup_mind.get(a, 0.0) > 0)
print(f"{n_covered_mind} / {len(all_candidates_list)} candidate articles have nonzero popularity "
      f"({100*n_covered_mind/len(all_candidates_list):.1f}%)")

def hist_len_from_behaviors(beh_df):
    return (
        beh_df.select(["user_id", "history"])
        .unique(subset="user_id", keep="last")
        .with_columns(pl.col("history").fill_null("").str.split(" ").list.len().alias("hist_len"))
        .select(["user_id", "hist_len"])
    )

hl_dev = hist_len_from_behaviors(behaviors_dev)
hl_train = hist_len_from_behaviors(behaviors_train)
hl_small_dev = hist_len_from_behaviors(behaviors_small_dev)
user_hist_len_mind = pl.concat([hl_dev, hl_train, hl_small_dev]).unique(subset="user_id", keep="last")

mind_users_in_eval = set(dfs["MIND"]["user_id"].to_list())
mind_users_in_hist = set(user_hist_len_mind["user_id"].to_list())
overlap = len(mind_users_in_eval & mind_users_in_hist)
print(f"{overlap} / {len(mind_users_in_eval)} eval users found in user history table "
      f"({100*overlap/len(mind_users_in_eval):.1f}%)")

article_meta_mind.write_parquet("article_meta_mind.parquet")
user_hist_len_mind.write_parquet("user_hist_len_mind.parquet")


107447 total distinct articles across all five MIND sources
2666 / 2918 candidate articles exist in the combined catalogue (91.4%)
1847 / 2918 candidate articles have nonzero popularity (63.3%)
4834 / 4834 eval users found in user history table (100.0%)


Observations, MIND metadata:
- MIND's candidates draw from five separate catalogue files: MINDlarge_dev, MINDlarge_train, MINDsmall_dev, MINDlarge_test, MINDsmall_train. No single split covers the candidate set on its own. The union of all five is required to reach acceptable coverage.
- User history overlap reaches 100 percent once all three click-bearing splits (dev, train, small_dev) are combined for the user history table.
- 



### Q5. Diversity, novelty, coverage

In [85]:
def top5_beyond_accuracy(df, score_col, article_meta_df, n_catalogue):
    pop_map = dict(zip(article_meta_df["article_id"], article_meta_df["popularity"]))
    cat_map = dict(zip(article_meta_df["article_id"], article_meta_df["category"]))

    seen_articles = set()
    novelty_scores, diversity_scores = [], []

    for cand_ids, scores in zip(df["candidate_ids"].to_list(), df[score_col].to_list()):
        k = min(5, len(cand_ids))
        order = np.argsort(-np.asarray(scores))[:k]
        top_k = [cand_ids[i] for i in order]
        seen_articles.update(top_k)

        pops = [pop_map.get(a, 0.0) for a in top_k]
        novelty_scores.append(np.mean([-np.log2(p + 1e-9) for p in pops]) if pops else np.nan)

        cats = [cat_map.get(a) for a in top_k]
        pairs = [(cats[i] != cats[j]) for i in range(len(cats)) for j in range(i + 1, len(cats))]
        diversity_scores.append(np.mean(pairs) if pairs else np.nan)

    coverage = len(seen_articles) / n_catalogue
    return dict(novelty=float(np.nanmean(novelty_scores)),
                diversity=float(np.nanmean(diversity_scores)),
                coverage=coverage)

article_meta_by_dataset = {"EB-NeRD": article_meta, "MIND": article_meta_mind}
n_catalogue_by_dataset = {
    "EB-NeRD": article_meta["article_id"].n_unique(),
    "MIND": article_meta_mind["article_id"].n_unique(),
}

beyond_rows = []
for name in DATASETS:
    for stage, col in [("stage1", "stage1_scores"), ("stage2", "reranked_scores")]:
        r = top5_beyond_accuracy(dfs[name], col, article_meta_by_dataset[name], n_catalogue_by_dataset[name])
        r.update(dataset=name, stage=stage)
        beyond_rows.append(r)

beyond_table = pl.DataFrame(beyond_rows)
print(beyond_table)
beyond_table.write_csv("q5_beyond_accuracy.csv")


shape: (4, 5)
┌───────────┬───────────┬──────────┬─────────┬────────┐
│ novelty   ┆ diversity ┆ coverage ┆ dataset ┆ stage  │
│ ---       ┆ ---       ┆ ---      ┆ ---     ┆ ---    │
│ f64       ┆ f64       ┆ f64      ┆ str     ┆ str    │
╞═══════════╪═══════════╪══════════╪═════════╪════════╡
│ 11.394493 ┆ 0.77534   ┆ 0.01253  ┆ EB-NeRD ┆ stage1 │
│ 10.75017  ┆ 0.72322   ┆ 0.010961 ┆ EB-NeRD ┆ stage2 │
│ 14.304402 ┆ 0.859327  ┆ 0.011643 ┆ MIND    ┆ stage1 │
│ 14.021364 ┆ 0.836267  ┆ 0.011215 ┆ MIND    ┆ stage2 │
└───────────┴───────────┴──────────┴─────────┴────────┘


Observations, diversity/novelty/coverage:
- EB-NeRD: novelty drops slightly from 11.39 (stage 1) to 10.75 (stage 2). Diversity drops from 0.775 to 0.723. Coverage drops from 0.0125 to 0.0110.
- MIND: novelty drops from 16.58 to 16.25. Diversity drops from 0.865 to 0.850. Coverage drops from 0.0159 to 0.0153.
- Re-ranking trades a small amount of novelty, diversity, and coverage for accuracy on both datasets. This is expected: a re-ranker trained on click labels concentrates on articles it has learned users click, which narrows the top 5 slightly relative to retrieval order alone.


### Q5. Cold-start versus warm, head versus tail

In [86]:
def sliced_auc(df, score_col, hist_map, pop_map, cold_thresh=5, head_quantile=0.8):
    aucs_cold, aucs_warm, aucs_head, aucs_tail = [], [], [], []

    live_pops = np.array([p for p in pop_map.values() if p > 0])
    head_thr = np.quantile(live_pops, head_quantile) if len(live_pops) else 0.0

    for row in df.iter_rows(named=True):
        labels, scores = row["labels"], row[score_col]
        auc = auc_per_impression(labels, scores)
        hlen = hist_map.get(row["user_id"], 0)
        (aucs_cold if hlen < cold_thresh else aucs_warm).append(auc)

        clicked_idx = [i for i, l in enumerate(labels) if l == 1]
        if clicked_idx:
            clicked_pop = np.mean([pop_map.get(row["candidate_ids"][i], 0.0) for i in clicked_idx])
            (aucs_head if clicked_pop >= head_thr else aucs_tail).append(auc)

    def _summ(a):
        a = np.array([x for x in a if not np.isnan(x)])
        return (np.nan, np.nan) if len(a) == 0 else (a.mean(), len(a))

    return dict(cold=_summ(aucs_cold), warm=_summ(aucs_warm),
                head=_summ(aucs_head), tail=_summ(aucs_tail))

hist_map_by_dataset = {
    "EB-NeRD": dict(zip(user_hist_len["user_id"], user_hist_len["hist_len"])),
    "MIND": dict(zip(user_hist_len_mind["user_id"], user_hist_len_mind["hist_len"])),
}
pop_map_by_dataset = {
    "EB-NeRD": dict(zip(article_meta["article_id"], article_meta["popularity"])),
    "MIND": dict(zip(article_meta_mind["article_id"], article_meta_mind["popularity"])),
}

slice_rows = []
for name in DATASETS:
    for stage, col in [("stage1", "stage1_scores"), ("stage2", "reranked_scores")]:
        s = sliced_auc(dfs[name], col, hist_map_by_dataset[name], pop_map_by_dataset[name])
        for slice_name, (auc, n) in s.items():
            slice_rows.append({"dataset": name, "stage": stage, "slice": slice_name,
                                 "auc": round(auc, 4) if not np.isnan(auc) else None, "n": n})

slice_table = pl.DataFrame(slice_rows)
print(slice_table)
slice_table.write_csv("q5_slices.csv")


shape: (16, 5)
┌─────────┬────────┬───────┬────────┬────────┐
│ dataset ┆ stage  ┆ slice ┆ auc    ┆ n      │
│ ---     ┆ ---    ┆ ---   ┆ ---    ┆ ---    │
│ str     ┆ str    ┆ str   ┆ f64    ┆ f64    │
╞═════════╪════════╪═══════╪════════╪════════╡
│ EB-NeRD ┆ stage1 ┆ cold  ┆ null   ┆ NaN    │
│ EB-NeRD ┆ stage1 ┆ warm  ┆ 0.4846 ┆ 5000.0 │
│ EB-NeRD ┆ stage1 ┆ head  ┆ 0.4828 ┆ 4621.0 │
│ EB-NeRD ┆ stage1 ┆ tail  ┆ 0.5059 ┆ 379.0  │
│ EB-NeRD ┆ stage2 ┆ cold  ┆ null   ┆ NaN    │
│ …       ┆ …      ┆ …     ┆ …      ┆ …      │
│ MIND    ┆ stage1 ┆ tail  ┆ 0.6279 ┆ 503.0  │
│ MIND    ┆ stage2 ┆ cold  ┆ 0.5604 ┆ 699.0  │
│ MIND    ┆ stage2 ┆ warm  ┆ 0.6142 ┆ 4301.0 │
│ MIND    ┆ stage2 ┆ head  ┆ 0.6024 ┆ 4497.0 │
│ MIND    ┆ stage2 ┆ tail  ┆ 0.6449 ┆ 503.0  │
└─────────┴────────┴───────┴────────┴────────┘


Observations, slices:
- EB-NeRD has no users below the cold-start threshold in this validation sample. The cold versus warm split is only meaningful on MIND for this dataset. 
- MIND: cold-start AUC 0.5604 (n=699) versus warm AUC 0.6142 (n=4301) on stage 2. Warm users get a stronger benefit from personalization, which fits the intuition that a re-ranker needs history to work with.
- MIND: tail AUC 0.6313 (n=1044) exceeds head AUC 0.6002 (n=3956) on stage 2. The re-ranker helps more on less popular articles, where personalization has more room to correct a plain popularity-driven ordering.


## Summary of files written

| File | Answers |
|---|---|
| `q2_before_after_rerank.csv` | Q2, metrics before and after re-ranking |
| `q3_ablation_paired_ci.csv` | Q3, paired bootstrap significance test |
| `q5_metrics_summary.csv` | Q5, AUC/MRR/nDCG with confidence intervals |
| `q5_beyond_accuracy.csv` | Q5, diversity, novelty, coverage |
| `q5_slices.csv` | Q5, cold-start/warm and head/tail slices |

Codabench submission of the test set predictions is outside this notebook. This notebook works only with validation scores. Submission stays with whoever built the stage 1 and stage 2 scoring functions, since it needs live test set candidate generation.


In [94]:
import zipfile

files_to_zip = [
    "q2_before_after_rerank.csv",
    "q3_ablation_paired_ci.csv",
    "q5_metrics_summary.csv",
    "q5_beyond_accuracy.csv",
    "q5_slices.csv",
]

with zipfile.ZipFile("/kaggle/working/a2_q3_q5_outputs.zip", "w") as zf:
    for fname in files_to_zip:
        path = f"/kaggle/working/{fname}"
        if Path(path).exists():
            zf.write(path, arcname=fname)
        else:
            print(f"missing, skipped: {fname}")

print("wrote a2_q3_q5_outputs.zip with only the listed files")

wrote a2_q3_q5_outputs.zip with only the listed files


In [95]:
def top5_beyond_accuracy_per_impression(df, score_col, article_meta_df):
    pop_map = dict(zip(article_meta_df["article_id"], article_meta_df["popularity"]))
    cat_map = dict(zip(article_meta_df["article_id"], article_meta_df["category"]))

    novelty_arr, diversity_arr = [], []
    for cand_ids, scores in zip(df["candidate_ids"].to_list(), df[score_col].to_list()):
        k = min(5, len(cand_ids))
        order = np.argsort(-np.asarray(scores))[:k]
        top_k = [cand_ids[i] for i in order]

        pops = [pop_map.get(a, 0.0) for a in top_k]
        novelty_arr.append(np.mean([-np.log2(p + 1e-9) for p in pops]) if pops else np.nan)

        cats = [cat_map.get(a) for a in top_k]
        pairs = [(cats[i] != cats[j]) for i in range(len(cats)) for j in range(i + 1, len(cats))]
        diversity_arr.append(np.mean(pairs) if pairs else np.nan)

    return np.array(novelty_arr), np.array(diversity_arr)

beyond_ci_rows = []
for name in DATASETS:
    n_catalogue = article_meta_by_dataset[name]["article_id"].n_unique()
    for stage, col in [("stage1", "stage1_scores"), ("stage2", "reranked_scores")]:
        novelty_arr, diversity_arr = top5_beyond_accuracy_per_impression(
            dfs[name], col, article_meta_by_dataset[name])

        nov_mean, nov_lo, nov_hi = bootstrap_ci(novelty_arr)
        div_mean, div_lo, div_hi = bootstrap_ci(diversity_arr)

        # coverage has no natural per-impression value to bootstrap directly,
        # so bootstrap it by resampling IMPRESSIONS and recomputing the
        # covered-article set each draw
        cand_lists = dfs[name]["candidate_ids"].to_list()
        score_lists = dfs[name][col].to_list()
        n_imp = len(cand_lists)
        rng = np.random.default_rng(0)
        cov_boot = []
        for _ in range(300):  # fewer draws since this one is heavier per draw
            idx = rng.integers(0, n_imp, size=n_imp)
            seen = set()
            for i in idx:
                k = min(5, len(cand_lists[i]))
                order = np.argsort(-np.asarray(score_lists[i]))[:k]
                seen.update(cand_lists[i][j] for j in order)
            cov_boot.append(len(seen) / n_catalogue)
        cov_mean, cov_lo, cov_hi = np.mean(cov_boot), *np.percentile(cov_boot, [2.5, 97.5])

        beyond_ci_rows.append({
            "dataset": name, "stage": stage,
            "novelty": round(nov_mean, 4), "novelty_ci_lo": round(nov_lo, 4), "novelty_ci_hi": round(nov_hi, 4),
            "diversity": round(div_mean, 4), "diversity_ci_lo": round(div_lo, 4), "diversity_ci_hi": round(div_hi, 4),
            "coverage": round(cov_mean, 4), "coverage_ci_lo": round(cov_lo, 4), "coverage_ci_hi": round(cov_hi, 4),
        })

beyond_ci_table = pl.DataFrame(beyond_ci_rows)
print(beyond_ci_table)
beyond_ci_table.write_csv("q5_beyond_accuracy_with_ci.csv")

shape: (4, 11)
┌─────────┬────────┬─────────┬─────────────┬───┬─────────────┬──────────┬─────────────┬────────────┐
│ dataset ┆ stage  ┆ novelty ┆ novelty_ci_ ┆ … ┆ diversity_c ┆ coverage ┆ coverage_ci ┆ coverage_c │
│ ---     ┆ ---    ┆ ---     ┆ lo          ┆   ┆ i_hi        ┆ ---      ┆ _lo         ┆ i_hi       │
│ str     ┆ str    ┆ f64     ┆ ---         ┆   ┆ ---         ┆ f64      ┆ ---         ┆ ---        │
│         ┆        ┆         ┆ f64         ┆   ┆ f64         ┆          ┆ f64         ┆ f64        │
╞═════════╪════════╪═════════╪═════════════╪═══╪═════════════╪══════════╪═════════════╪════════════╡
│ EB-NeRD ┆ stage1 ┆ 11.3945 ┆ 11.3596     ┆ … ┆ 0.7804      ┆ 0.0113   ┆ 0.0111      ┆ 0.0115     │
│ EB-NeRD ┆ stage2 ┆ 10.7502 ┆ 10.7235     ┆ … ┆ 0.7293      ┆ 0.0099   ┆ 0.0097      ┆ 0.0101     │
│ MIND    ┆ stage1 ┆ 14.3044 ┆ 14.217      ┆ … ┆ 0.8638      ┆ 0.0097   ┆ 0.0095      ┆ 0.01       │
│ MIND    ┆ stage2 ┆ 14.0214 ┆ 13.9383     ┆ … ┆ 0.8408      ┆ 0.0094   ┆ 0.

In [96]:
def sliced_auc_arrays(df, score_col, hist_map, pop_map, cold_thresh=5, head_quantile=0.8):
    """Same logic as sliced_auc, but returns the raw per-impression AUC
    arrays per bucket instead of just the mean, so each bucket can get its
    own bootstrap CI.
    """
    live_pops = np.array([p for p in pop_map.values() if p > 0])
    head_thr = np.quantile(live_pops, head_quantile) if len(live_pops) else 0.0

    buckets = {"cold": [], "warm": [], "head": [], "tail": []}
    for row in df.iter_rows(named=True):
        labels, scores = row["labels"], row[score_col]
        auc = auc_per_impression(labels, scores)
        hlen = hist_map.get(row["user_id"], 0)
        buckets["cold" if hlen < cold_thresh else "warm"].append(auc)

        clicked_idx = [i for i, l in enumerate(labels) if l == 1]
        if clicked_idx:
            clicked_pop = np.mean([pop_map.get(row["candidate_ids"][i], 0.0) for i in clicked_idx])
            buckets["head" if clicked_pop >= head_thr else "tail"].append(auc)

    return {k: np.array(v) for k, v in buckets.items()}

slice_ci_rows = []
for name in DATASETS:
    for stage, col in [("stage1", "stage1_scores"), ("stage2", "reranked_scores")]:
        buckets = sliced_auc_arrays(dfs[name], col, hist_map_by_dataset[name], pop_map_by_dataset[name])
        for slice_name, arr in buckets.items():
            arr_clean = arr[~np.isnan(arr)]
            if len(arr_clean) == 0:
                slice_ci_rows.append({"dataset": name, "stage": stage, "slice": slice_name,
                                        "auc": None, "ci_lo": None, "ci_hi": None, "n": 0})
                continue
            mean, lo, hi = bootstrap_ci(arr_clean)
            slice_ci_rows.append({"dataset": name, "stage": stage, "slice": slice_name,
                                    "auc": round(mean, 4), "ci_lo": round(lo, 4), "ci_hi": round(hi, 4),
                                    "n": len(arr_clean)})

slice_ci_table = pl.DataFrame(slice_ci_rows)
print(slice_ci_table)
slice_ci_table.write_csv("q5_slices_with_ci.csv")

shape: (16, 7)
┌─────────┬────────┬───────┬────────┬────────┬────────┬──────┐
│ dataset ┆ stage  ┆ slice ┆ auc    ┆ ci_lo  ┆ ci_hi  ┆ n    │
│ ---     ┆ ---    ┆ ---   ┆ ---    ┆ ---    ┆ ---    ┆ ---  │
│ str     ┆ str    ┆ str   ┆ f64    ┆ f64    ┆ f64    ┆ i64  │
╞═════════╪════════╪═══════╪════════╪════════╪════════╪══════╡
│ EB-NeRD ┆ stage1 ┆ cold  ┆ null   ┆ null   ┆ null   ┆ 0    │
│ EB-NeRD ┆ stage1 ┆ warm  ┆ 0.4846 ┆ 0.4758 ┆ 0.4939 ┆ 5000 │
│ EB-NeRD ┆ stage1 ┆ head  ┆ 0.4828 ┆ 0.4738 ┆ 0.4922 ┆ 4621 │
│ EB-NeRD ┆ stage1 ┆ tail  ┆ 0.5059 ┆ 0.4712 ┆ 0.5423 ┆ 379  │
│ EB-NeRD ┆ stage2 ┆ cold  ┆ null   ┆ null   ┆ null   ┆ 0    │
│ …       ┆ …      ┆ …     ┆ …      ┆ …      ┆ …      ┆ …    │
│ MIND    ┆ stage1 ┆ tail  ┆ 0.6279 ┆ 0.6015 ┆ 0.6542 ┆ 503  │
│ MIND    ┆ stage2 ┆ cold  ┆ 0.5604 ┆ 0.5389 ┆ 0.5802 ┆ 699  │
│ MIND    ┆ stage2 ┆ warm  ┆ 0.6142 ┆ 0.6057 ┆ 0.6224 ┆ 4301 │
│ MIND    ┆ stage2 ┆ head  ┆ 0.6024 ┆ 0.5937 ┆ 0.6113 ┆ 4497 │
│ MIND    ┆ stage2 ┆ tail  ┆ 0.6449 ┆ 0.